#### Check cleaned CME Data

In [3]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent
df = pd.read_parquet(project_root / 'data' / 'cleaned' / 'cme_options_clean.parquet')

# Simulate max_tau = 180 days
df_180 = df[df['days_to_expiry'] <= 180].copy()

print(f"Rows with τ ≤ 365: {len(df):,}")
print(f"Rows with τ ≤ 180: {len(df_180):,}")
print(f"Dropped: {len(df) - len(df_180):,} ({(len(df) - len(df_180))/len(df):.1%})")

# Repeat the diagnostics
per_slice = df_180.groupby(['date', 'expiration']).size().reset_index(name='n_options')
print(f"\nOptions per (date, expiration) slice:")
print(per_slice['n_options'].describe())
print(f"\nSlices with < 4 options: {(per_slice['n_options'] < 4).sum()} ({(per_slice['n_options'] < 4).mean():.1%})")
print(f"Slices with < 3 options: {(per_slice['n_options'] < 3).sum()} ({(per_slice['n_options'] < 3).mean():.1%})")

exp_per_day = df_180.groupby('date')['expiration'].nunique()
print(f"\nExpirations per day: median={exp_per_day.median():.0f}, min={exp_per_day.min()}")

daily = df_180.groupby('date').size()
thin_days = daily[daily < 10]
print(f"\nDays with < 10 total options: {len(thin_days)}")

Rows with τ ≤ 365: 66,072
Rows with τ ≤ 180: 65,598
Dropped: 474 (0.7%)

Options per (date, expiration) slice:
count    2804.000000
mean       23.394437
std        17.850788
min         1.000000
25%         9.000000
50%        21.000000
75%        34.000000
max       148.000000
Name: n_options, dtype: float64

Slices with < 4 options: 353 (12.6%)
Slices with < 3 options: 283 (10.1%)

Expirations per day: median=3, min=1

Days with < 10 total options: 31
